# 04b - Generate order items

Business objective: orders alone only tells us order counts. To answer
"why did sales drop" we need actual revenue - quantity x unit price per line
item, summed per order. This notebook was added because the original
8-notebook plan omitted it while still requiring revenue analysis downstream.

What we're generating: 1-4 line items per order, referencing a real product
at that product's price, with occasional small discounts.

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np

rng = np.random.default_rng(cfg.SEED + 7)

orders = pd.read_csv(cfg.DATA_DIR / "orders.csv")
products = pd.read_csv(cfg.DATA_DIR / "products.csv")
print(f"Loaded {len(orders):,} orders and {len(products):,} products")

Loaded 35,591 orders and 200 products


## Generation logic

In [2]:
n_items_per_order = rng.choice([1, 2, 3, 4], size=len(orders), p=[0.45, 0.30, 0.15, 0.10])
product_ids = products["product_id"].values
price_lookup = products.set_index("product_id")["unit_price"].to_dict()

rows = []
item_id = 1
for order_id, n_items in zip(orders["order_id"].values, n_items_per_order):
    chosen = rng.choice(product_ids, size=n_items, replace=False)
    for pid in chosen:
        qty = int(rng.choice([1, 1, 1, 2, 2, 3]))
        discount = rng.choice([1.0, 1.0, 1.0, 0.9, 0.8], p=[0.7, 0.1, 0.1, 0.06, 0.04])
        unit_price = round(price_lookup[pid] * discount, 2)
        rows.append({
            "order_item_id": item_id,
            "order_id": order_id,
            "product_id": int(pid),
            "quantity": qty,
            "unit_price": unit_price,
            "line_total": round(unit_price * qty, 2),
        })
        item_id += 1

order_items = pd.DataFrame(rows)
print(f"Generated {len(order_items):,} order items")
order_items.head()

Generated 67,743 order items


,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,11,1,28.55,28.55
1,2,2,43,1,8.83,8.83
2,3,2,159,1,87.96,87.96
3,4,3,51,1,764.02,764.02
4,5,4,144,1,20.50,20.50


## Sanity check: Sydney revenue vs Melbourne revenue

In [3]:
stores = pd.read_csv(cfg.DATA_DIR / "stores.csv")
o = pd.read_csv(cfg.DATA_DIR / "orders.csv", parse_dates=["order_date"]).merge(stores[["store_id","city"]], on="store_id")
rev = order_items.merge(o[["order_id","city","order_date"]], on="order_id")
rev["month"] = rev["order_date"].dt.to_period("M").astype(str)
rev.groupby(["month","city"])["line_total"].sum().unstack(fill_value=0)[["Sydney","Melbourne"]].tail(4)

city,Sydney,Melbourne
month,,
2026-04,188404.20,219048.00
2026-05,96932.80,232955.96
2026-06,118688.60,224356.84
2026-07,1858.03,7180.36


## Validation

In [4]:
assert order_items["order_item_id"].is_unique
assert order_items["order_id"].isin(orders["order_id"]).all()
assert order_items["product_id"].isin(products["product_id"]).all()
assert (order_items["unit_price"] > 0).all()
assert order_items.isnull().sum().sum() == 0
print("All validation checks passed.")

All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "order_items.csv"
order_items.to_csv(out_path, index=False)
print(f"Wrote {len(order_items):,} rows to {out_path}")

Wrote 67,743 rows to /home/claude/project/eaida/data/raw/order_items.csv


## Summary

Generated order-level line items with realistic quantity and discount
patterns, giving an actual revenue figure per store/month and confirming the
Sydney drop is a real dollar decline, not just an order-count artifact.
Saved to `data/raw/order_items.csv`.